# GEC Pipeline - Training and Inference

This notebook walks through the complete GEC (Grammatical Error Correction) pipeline:
1. Data preparation and feature extraction
2. Training the edit tagger model
3. Running inference on new text

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

print("Setup complete!")

Setup complete!


## Part 1: Data Preparation

In [2]:
from src.services.gec.config import (
    TRAIN_SENT_PATH,
    TRAIN_COR_PATH,
    LABEL2ID_PATH,
    ID2LABEL_PATH,
    CHECKPOINT_PATH,
)

# print("Checking data files...")
# print(f"Training sentences: {TRAIN_SENT_PATH.exists()}")
# print(f"Training corrections: {TRAIN_COR_PATH.exists()}")
# print(f"Checkpoint (processed data): {CHECKPOINT_PATH.exists()}")

In [3]:
import json

# from src.services.gec.features.build_train import build_train

# if CHECKPOINT_PATH.exists():
#     print("Loading existing processed data...")
#     with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
#         num_lines = sum(1 for _ in f)
#     print(f"✓ Found {num_lines} training examples")
    
#     with open(LABEL2ID_PATH, 'r', encoding='utf-8') as f:
#         label2id = json.load(f)
#     print(f"✓ Label vocabulary: {len(label2id)} labels")
# else:
#     print("⚠️  No processed data found. Run build_train() first.")
#     build_train()

In [4]:
# print("Sample training example:")
# with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
#     first_line = f.readline()
#     example = json.loads(first_line.strip())
#     print(f"Subwords (first 15): {example['subwords'][:15]}")
#     print(f"Labels (first 15): {example['labels'][:15]}")
#     print(f"Total subwords: {len(example['subwords'])}")
#     print(f"Total labels: {len(example['labels'])}")
#     print(f"Length match: {len(example['subwords']) == len(example['labels'])}")

## Part 2: Model Training

In [5]:
from transformers import AutoTokenizer
from src.services.gec.training.datasets import GECTrainingDataset
from src.services.gec.training.trainer import build_trainer

MODEL_CHECKPOINT = "aubmindlab/bert-base-arabertv02"
OUTPUT_DIR = Path("./gec_models/edit_tagger_v1")
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
MAX_LENGTH = 256

# print(f"Model: {MODEL_CHECKPOINT}")
# print(f"Output: {OUTPUT_DIR}")
# print(f"Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, Max length: {MAX_LENGTH}")

/home/somia/baligh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import json

with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    loaded_label2id = json.load(f)

with open(ID2LABEL_PATH, "r", encoding="utf-8") as f:
    loaded_id2label = json.load(f)

label2id = {
    str(k): int(v)
    for k, v in loaded_label2id.items()
}

id2label = {
    int(k): str(v)
    for k, v in loaded_id2label.items()
}

print(type(next(iter(id2label.keys()))))
print(type(next(iter(label2id.keys()))))

<class 'int'>
<class 'str'>


In [7]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✓ Vocab size: {tokenizer.vocab_size}")
print(f"✓ PAD token: {tokenizer.pad_token}")
print(f"✓ PAD token ID: {tokenizer.pad_token_id}")

Loading tokenizer...


✓ Vocab size: 64000
✓ PAD token: [PAD]
✓ PAD token ID: 0


In [8]:
print("Loading training dataset...")
train_dataset = GECTrainingDataset(
    jsonl_path=CHECKPOINT_PATH,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=MAX_LENGTH,
)
print(f"✓ Training examples: {len(train_dataset)}")

sample = train_dataset[0]
print(f"\nSample item structure:")
print(f"  input_ids length: {len(sample['input_ids'])}")
print(f"  attention_mask length: {len(sample['attention_mask'])}")
print(f"  labels length: {len(sample['labels'])}")
print(f"  Lengths match: {len(sample['input_ids']) == len(sample['labels'])}")

Loading training dataset...
✓ Training examples: 5000

Sample item structure:
  input_ids length: 80
  attention_mask length: 80
  labels length: 80
  Lengths match: True


In [9]:
from src.services.gec.training.model import create_model
print("Initializing model...")
model = create_model(
    checkpoint=MODEL_CHECKPOINT,
    label2id=label2id,
)
print(f"✓ Model initialized with {len(label2id)} output labels")

Initializing model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2350.37it/s]
[transformers] BertForTokenClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/ar

✓ Model initialized with 1534 output labels


In [10]:
print("Building trainer...")
# trainer = build_trainer(
#     model=model,
#     tokenizer= tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=None,
#     output_dir=OUTPUT_DIR,
#     num_train_epochs=NUM_EPOCHS,
#     learning_rate=LEARNING_RATE,
#     fp16=False,
#     label2id_path=LABEL2ID_PATH,
# )
print("✓ Trainer ready!")

Building trainer...
✓ Trainer ready!


In [11]:
print("\n🚀 Starting training...")
print("="*60)

# trainer.train()

print("\n" + "="*60)    
print("✓ Training complete!")


🚀 Starting training...

✓ Training complete!


## Part 3: Inference

In [12]:
from transformers import AutoModelForTokenClassification
from src.services.gec.modules.edit_tagger.inference import GECInferencePipeline
from src.services.gec.utils.string_utils import Tokenizer


best_model_path = Path("./gec_models/edit_tagger_v1/checkpoint-939")
if not best_model_path.exists():
    print("⚠️  Model not found. Please train first.")
else:
    print(f"Loading model from {best_model_path}...")
    inference_model = AutoModelForTokenClassification.from_pretrained(best_model_path)
    print(f"✓ Model loaded ({inference_model.config.num_labels} labels)")

Loading model from gec_models/edit_tagger_v1/checkpoint-939...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1606.59it/s]

✓ Model loaded (1534 labels)


In [13]:
class LabelVocab:
    def __init__(self, id2label):
        self.id2label = id2label

class DummyRewriter:
    pass

if best_model_path.exists():
    tokenizer = Tokenizer()
    label_vocab = LabelVocab(id2label)
    rewriter = DummyRewriter()

    pipeline = GECInferencePipeline(
        model=inference_model,
        tokenizer=tokenizer,
        label_vocab=label_vocab,
    )
    print("✓ Inference pipeline ready!")

✓ Inference pipeline ready!


In [14]:
subwords, labels = pipeline.predict("اكيد")

print(subwords)
print(labels)

{'input_ids': tensor([[   2, 7291,  334,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}
<class 'int'>
<class 'int'>
[800, 1284, 800, 800]
1284
800
1534
1534
['اك', '##يد']
['R_[أ]K*', 'K*']


In [15]:
print("Sample training example:")
with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    example = json.loads(first_line.strip())
    print(f"Subwords (first 15): {example['subwords'][:15]}")
    print(f"Labels - count version (first 15): {example.get('labels', [])[:15]}")
    print(f"Labels - star version (first 15): {example.get('labels_star', [])[:15]}")
    print(f"\nUsing labels_star for training: {len(example.get('labels_star', []))} labels")
    print(f"Total subwords: {len(example['subwords'])}")
    print(f"Length match: {len(example['subwords']) == len(example.get('labels_star', []))}")

Sample training example:
Subwords (first 15): ['الى', 'التعليق', 'رقم', '2', ' ', 'اك', '##يد', 'ان', 'لحكام', 'العرب', 'والمسلمين', 'مسؤولية', 'يتمثل', 'اد', '##ناها']
Labels - count version (first 15): ['R_[إ]K2', 'K7', 'K3', 'K', 'R_[:]', 'R_[أ]K', 'K2', 'R_[أ]K', 'I_[ل]K5', 'K5', 'K9', 'K7', 'K5', 'R_[أ]K', 'K4']
Labels - star version (first 15): ['R_[إ]K*', 'K*', 'K*', 'K', 'R_[:]', 'R_[أ]K', 'K*', 'R_[أ]K', 'I_[ل]K*', 'K*', 'K*', 'K*', 'K*', 'R_[أ]K', 'K*']

Using labels_star for training: 56 labels
Total subwords: 56
Length match: True
